# 11. Dataclasses, value objects, and copying

**Level:** Intermediate

**Learning outcome:** Choose field-based equality, immutable records, factories, ordering, and appropriate copy depth.

**Prerequisites:** 01-03: identity, attributes, invariants, and mutable containers.

[Learning-path index](00_START_HERE_OOP_Learning_Path.ipynb) | [Previous](10_Dependency_Inversion_and_Policy_Contracts.ipynb) | [Next notebook](12_Special_Methods_and_Container_Protocols.ipynb)

## How to study this notebook
1. Read the mental model and explanation before executing a code cell.
2. Predict the output, then run the cell and compare the stored result.
3. Trace constructor calls, attribute reads/writes, returned values, and exception paths.
4. Change one input, predict the difference, and keep meaningful assertions.
5. Complete the practice prompts and answer the final checkpoint aloud.

**Run independently:** every worked code cell includes its own imports, definitions, and input data. You do not need to run another notebook first. Use Python 3.11+ and a Python Jupyter kernel. Only the standard library is used. Async cells use top-level await, not asyncio.run inside Jupyter.

**Offline safety:** all network states, commands, approvals, and failures are simulations. The code does not connect to devices, use credentials, write device configuration, or need external input files. Teaching limits and command strings are not vendor specifications. Assertions are learning checks, not production input validation.

## Concept notes and mental model

**Level:** Intermediate

**Learning goal:** Choose value semantics, safe factories, immutable fields, and deliberate copy depth.

**Dataclasses** generate common methods from annotated fields. A normal dataclass usually gets
initialization, representation, and field-based equality. This reduces boilerplate for records,
but it does not validate types, parse addresses, or make every domain model correct. Use
`__post_init__` for additional validation after generated initialization.

**Identity versus value.** Two separately created route records may compare equal because their
fields match while `first is second` remains false. Choose equality based on domain meaning.
A live session is often an identity-bearing resource; a route key is often a value.

**Mutable fields** should use `field(default_factory=list)` or the appropriate factory, which
creates a fresh container per instance. `frozen=True` blocks normal field reassignment but is
not deep immutability: a list stored in a frozen object can still change. Prefer immutable
nested values when you need a dependable dictionary key.

**Ordering** is opt-in and follows the dataclass field order for fields included in comparison.
Put the intended sort key first or define an explicit key function. Avoid accidental ordering
by a hostname when your intent was route preference.

**Copy choices.** Assignment shares the same object. A shallow copy creates a new outer object
while nested references remain shared. A deep copy recursively copies suitable nested values,
subject to customization and memoization. Neither approach makes a meaningful duplicate of a
live network session, lock, or external resource. For a frozen configuration record,
`dataclasses.replace` expresses a changed value clearly without mutating the original.

**Pause and explain:** Explain frozen versus deep immutability and predict shallow-copy behavior for a nested list.

## Worked-example structure

This notebook contains **5 worked patterns and 50 executable scenario examples**. Each pattern explains its purpose, logic, common mistakes, and extension exercise. Its ten scenarios vary the input data; they are related practice, not ten unrelated algorithms. Original Example IDs are retained so you can locate the same exercise in `AI_oops_learning.ipynb`.

The input table lists only fields needed by that pattern. Expected output is shown before each code cell and a saved executed result appears afterward. Expected rejection branches are checked explicitly; unexpected exceptions are not silently ignored.

## Pattern 031: Dataclass value equality

**Core concept:** Generated equality compares fields rather than requiring identical objects.

**Network-engineering purpose:** Two independently parsed route keys can represent the same logical destination.

### Read the implementation in this order
1. Declare annotated fields on a frozen dataclass.
2. Construct two equal records independently.
3. Compare them with == and check their hashes.
4. Check that identity remains different and a set deduplicates equal values.

**Common mistake / boundary:** Field equality may be wrong for resources with identity or lifecycle; choose semantics deliberately.

**Practice after the ten worked scenarios:** Add a non-key field with compare=False and reason about its effect on equality.

The following ten examples intentionally reuse this implementation with explicit local scenario data. The displayed output comes from running each full code cell; assertions also check behavior, including expected rejections where present.

### Example 0301 - Dataclass value equality: branch scenario

**Scenario 01/10; pattern 031; Intermediate.** This is a self-contained offline exercise. Two independently parsed route keys can represent the same logical destination.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `site` | `'branch'` |
| `prefix` | `'192.0.2.0/28'` |

**Logic to trace:** Declare annotated fields on a frozen dataclass. Check that identity remains different and a set deduplicates equal values. Follow the class-specific steps under Pattern 031 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
RouteKey(prefix='192.0.2.0/28', vrf='branch')
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Field equality may be wrong for resources with identity or lifecycle; choose semantics deliberately.

In [1]:
# Example 0301: Dataclass value equality - branch
case = {'site': 'branch', 'prefix': '192.0.2.0/28'}

from dataclasses import dataclass

@dataclass(frozen=True)
class RouteKey:
    prefix: str
    vrf: str

first = RouteKey(case["prefix"], case["site"])
second = RouteKey(case["prefix"], case["site"])
assert first == second
assert first is not second
assert hash(first) == hash(second)
assert len({first, second}) == 1
print(first)


RouteKey(prefix='192.0.2.0/28', vrf='branch')


### Example 0302 - Dataclass value equality: campus scenario

**Scenario 02/10; pattern 031; Intermediate.** This is a self-contained offline exercise. Two independently parsed route keys can represent the same logical destination.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `site` | `'campus'` |
| `prefix` | `'192.0.2.16/28'` |

**Logic to trace:** Declare annotated fields on a frozen dataclass. Check that identity remains different and a set deduplicates equal values. Follow the class-specific steps under Pattern 031 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
RouteKey(prefix='192.0.2.16/28', vrf='campus')
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Field equality may be wrong for resources with identity or lifecycle; choose semantics deliberately.

In [2]:
# Example 0302: Dataclass value equality - campus
case = {'site': 'campus', 'prefix': '192.0.2.16/28'}

from dataclasses import dataclass

@dataclass(frozen=True)
class RouteKey:
    prefix: str
    vrf: str

first = RouteKey(case["prefix"], case["site"])
second = RouteKey(case["prefix"], case["site"])
assert first == second
assert first is not second
assert hash(first) == hash(second)
assert len({first, second}) == 1
print(first)


RouteKey(prefix='192.0.2.16/28', vrf='campus')


### Example 0303 - Dataclass value equality: dc scenario

**Scenario 03/10; pattern 031; Intermediate.** This is a self-contained offline exercise. Two independently parsed route keys can represent the same logical destination.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `site` | `'dc'` |
| `prefix` | `'192.0.2.32/28'` |

**Logic to trace:** Declare annotated fields on a frozen dataclass. Check that identity remains different and a set deduplicates equal values. Follow the class-specific steps under Pattern 031 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
RouteKey(prefix='192.0.2.32/28', vrf='dc')
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Field equality may be wrong for resources with identity or lifecycle; choose semantics deliberately.

In [3]:
# Example 0303: Dataclass value equality - dc
case = {'site': 'dc', 'prefix': '192.0.2.32/28'}

from dataclasses import dataclass

@dataclass(frozen=True)
class RouteKey:
    prefix: str
    vrf: str

first = RouteKey(case["prefix"], case["site"])
second = RouteKey(case["prefix"], case["site"])
assert first == second
assert first is not second
assert hash(first) == hash(second)
assert len({first, second}) == 1
print(first)


RouteKey(prefix='192.0.2.32/28', vrf='dc')


### Example 0304 - Dataclass value equality: wan scenario

**Scenario 04/10; pattern 031; Intermediate.** This is a self-contained offline exercise. Two independently parsed route keys can represent the same logical destination.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `site` | `'wan'` |
| `prefix` | `'192.0.2.48/28'` |

**Logic to trace:** Declare annotated fields on a frozen dataclass. Check that identity remains different and a set deduplicates equal values. Follow the class-specific steps under Pattern 031 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
RouteKey(prefix='192.0.2.48/28', vrf='wan')
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Field equality may be wrong for resources with identity or lifecycle; choose semantics deliberately.

In [4]:
# Example 0304: Dataclass value equality - wan
case = {'site': 'wan', 'prefix': '192.0.2.48/28'}

from dataclasses import dataclass

@dataclass(frozen=True)
class RouteKey:
    prefix: str
    vrf: str

first = RouteKey(case["prefix"], case["site"])
second = RouteKey(case["prefix"], case["site"])
assert first == second
assert first is not second
assert hash(first) == hash(second)
assert len({first, second}) == 1
print(first)


RouteKey(prefix='192.0.2.48/28', vrf='wan')


### Example 0305 - Dataclass value equality: lab scenario

**Scenario 05/10; pattern 031; Intermediate.** This is a self-contained offline exercise. Two independently parsed route keys can represent the same logical destination.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `site` | `'lab'` |
| `prefix` | `'192.0.2.64/28'` |

**Logic to trace:** Declare annotated fields on a frozen dataclass. Check that identity remains different and a set deduplicates equal values. Follow the class-specific steps under Pattern 031 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
RouteKey(prefix='192.0.2.64/28', vrf='lab')
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Field equality may be wrong for resources with identity or lifecycle; choose semantics deliberately.

In [5]:
# Example 0305: Dataclass value equality - lab
case = {'site': 'lab', 'prefix': '192.0.2.64/28'}

from dataclasses import dataclass

@dataclass(frozen=True)
class RouteKey:
    prefix: str
    vrf: str

first = RouteKey(case["prefix"], case["site"])
second = RouteKey(case["prefix"], case["site"])
assert first == second
assert first is not second
assert hash(first) == hash(second)
assert len({first, second}) == 1
print(first)


RouteKey(prefix='192.0.2.64/28', vrf='lab')


### Example 0306 - Dataclass value equality: wireless scenario

**Scenario 06/10; pattern 031; Intermediate.** This is a self-contained offline exercise. Two independently parsed route keys can represent the same logical destination.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `site` | `'wireless'` |
| `prefix` | `'192.0.2.80/28'` |

**Logic to trace:** Declare annotated fields on a frozen dataclass. Check that identity remains different and a set deduplicates equal values. Follow the class-specific steps under Pattern 031 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
RouteKey(prefix='192.0.2.80/28', vrf='wireless')
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Field equality may be wrong for resources with identity or lifecycle; choose semantics deliberately.

In [6]:
# Example 0306: Dataclass value equality - wireless
case = {'site': 'wireless', 'prefix': '192.0.2.80/28'}

from dataclasses import dataclass

@dataclass(frozen=True)
class RouteKey:
    prefix: str
    vrf: str

first = RouteKey(case["prefix"], case["site"])
second = RouteKey(case["prefix"], case["site"])
assert first == second
assert first is not second
assert hash(first) == hash(second)
assert len({first, second}) == 1
print(first)


RouteKey(prefix='192.0.2.80/28', vrf='wireless')


### Example 0307 - Dataclass value equality: remote scenario

**Scenario 07/10; pattern 031; Intermediate.** This is a self-contained offline exercise. Two independently parsed route keys can represent the same logical destination.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `site` | `'remote'` |
| `prefix` | `'192.0.2.96/28'` |

**Logic to trace:** Declare annotated fields on a frozen dataclass. Check that identity remains different and a set deduplicates equal values. Follow the class-specific steps under Pattern 031 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
RouteKey(prefix='192.0.2.96/28', vrf='remote')
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Field equality may be wrong for resources with identity or lifecycle; choose semantics deliberately.

In [7]:
# Example 0307: Dataclass value equality - remote
case = {'site': 'remote', 'prefix': '192.0.2.96/28'}

from dataclasses import dataclass

@dataclass(frozen=True)
class RouteKey:
    prefix: str
    vrf: str

first = RouteKey(case["prefix"], case["site"])
second = RouteKey(case["prefix"], case["site"])
assert first == second
assert first is not second
assert hash(first) == hash(second)
assert len({first, second}) == 1
print(first)


RouteKey(prefix='192.0.2.96/28', vrf='remote')


### Example 0308 - Dataclass value equality: core scenario

**Scenario 08/10; pattern 031; Intermediate.** This is a self-contained offline exercise. Two independently parsed route keys can represent the same logical destination.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `site` | `'core'` |
| `prefix` | `'192.0.2.112/28'` |

**Logic to trace:** Declare annotated fields on a frozen dataclass. Check that identity remains different and a set deduplicates equal values. Follow the class-specific steps under Pattern 031 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
RouteKey(prefix='192.0.2.112/28', vrf='core')
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Field equality may be wrong for resources with identity or lifecycle; choose semantics deliberately.

In [8]:
# Example 0308: Dataclass value equality - core
case = {'site': 'core', 'prefix': '192.0.2.112/28'}

from dataclasses import dataclass

@dataclass(frozen=True)
class RouteKey:
    prefix: str
    vrf: str

first = RouteKey(case["prefix"], case["site"])
second = RouteKey(case["prefix"], case["site"])
assert first == second
assert first is not second
assert hash(first) == hash(second)
assert len({first, second}) == 1
print(first)


RouteKey(prefix='192.0.2.112/28', vrf='core')


### Example 0309 - Dataclass value equality: mgmt scenario

**Scenario 09/10; pattern 031; Intermediate.** This is a self-contained offline exercise. Two independently parsed route keys can represent the same logical destination.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `site` | `'mgmt'` |
| `prefix` | `'192.0.2.128/28'` |

**Logic to trace:** Declare annotated fields on a frozen dataclass. Check that identity remains different and a set deduplicates equal values. Follow the class-specific steps under Pattern 031 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
RouteKey(prefix='192.0.2.128/28', vrf='mgmt')
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Field equality may be wrong for resources with identity or lifecycle; choose semantics deliberately.

In [9]:
# Example 0309: Dataclass value equality - mgmt
case = {'site': 'mgmt', 'prefix': '192.0.2.128/28'}

from dataclasses import dataclass

@dataclass(frozen=True)
class RouteKey:
    prefix: str
    vrf: str

first = RouteKey(case["prefix"], case["site"])
second = RouteKey(case["prefix"], case["site"])
assert first == second
assert first is not second
assert hash(first) == hash(second)
assert len({first, second}) == 1
print(first)


RouteKey(prefix='192.0.2.128/28', vrf='mgmt')


### Example 0310 - Dataclass value equality: backup scenario

**Scenario 10/10; pattern 031; Intermediate.** This is a self-contained offline exercise. Two independently parsed route keys can represent the same logical destination.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `site` | `'backup'` |
| `prefix` | `'192.0.2.144/28'` |

**Logic to trace:** Declare annotated fields on a frozen dataclass. Check that identity remains different and a set deduplicates equal values. Follow the class-specific steps under Pattern 031 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
RouteKey(prefix='192.0.2.144/28', vrf='backup')
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Field equality may be wrong for resources with identity or lifecycle; choose semantics deliberately.

In [10]:
# Example 0310: Dataclass value equality - backup
case = {'site': 'backup', 'prefix': '192.0.2.144/28'}

from dataclasses import dataclass

@dataclass(frozen=True)
class RouteKey:
    prefix: str
    vrf: str

first = RouteKey(case["prefix"], case["site"])
second = RouteKey(case["prefix"], case["site"])
assert first == second
assert first is not second
assert hash(first) == hash(second)
assert len({first, second}) == 1
print(first)


RouteKey(prefix='192.0.2.144/28', vrf='backup')


## Pattern 032: Fresh dataclass collection fields

**Core concept:** default_factory builds a separate mutable value for each instance.

**Network-engineering purpose:** Each device's change log must remain independent.

### Read the implementation in this order
1. Use field(default_factory=list).
2. Create two records without supplying the log.
3. Append a change to the first.
4. Assert the second remains empty and the lists have different identities.

**Common mistake / boundary:** A factory is called for each needed default; passing one existing list explicitly can still share it.

**Practice after the ten worked scenarios:** Give the class a method that appends a timestamp-free structured event.

The following ten examples intentionally reuse this implementation with explicit local scenario data. The displayed output comes from running each full code cell; assertions also check behavior, including expected rejections where present.

### Example 0311 - Fresh dataclass collection fields: branch scenario

**Scenario 01/10; pattern 032; Intermediate.** This is a self-contained offline exercise. Each device's change log must remain independent.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'branch-edge-01'` |

**Logic to trace:** Use field(default_factory=list). Assert the second remains empty and the lists have different identities. Follow the class-specific steps under Pattern 032 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['reviewed'] []
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A factory is called for each needed default; passing one existing list explicitly can still share it.

In [11]:
# Example 0311: Fresh dataclass collection fields - branch
case = {'hostname': 'branch-edge-01'}

from dataclasses import dataclass, field

@dataclass
class Device:
    hostname: str
    changes: list[str] = field(default_factory=list)

first = Device(case["hostname"])
second = Device("spare")
first.changes.append("reviewed")
assert second.changes == []
assert first.changes is not second.changes
print(first.changes, second.changes)


['reviewed'] []


### Example 0312 - Fresh dataclass collection fields: campus scenario

**Scenario 02/10; pattern 032; Intermediate.** This is a self-contained offline exercise. Each device's change log must remain independent.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'campus-access-02'` |

**Logic to trace:** Use field(default_factory=list). Assert the second remains empty and the lists have different identities. Follow the class-specific steps under Pattern 032 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['reviewed'] []
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A factory is called for each needed default; passing one existing list explicitly can still share it.

In [12]:
# Example 0312: Fresh dataclass collection fields - campus
case = {'hostname': 'campus-access-02'}

from dataclasses import dataclass, field

@dataclass
class Device:
    hostname: str
    changes: list[str] = field(default_factory=list)

first = Device(case["hostname"])
second = Device("spare")
first.changes.append("reviewed")
assert second.changes == []
assert first.changes is not second.changes
print(first.changes, second.changes)


['reviewed'] []


### Example 0313 - Fresh dataclass collection fields: dc scenario

**Scenario 03/10; pattern 032; Intermediate.** This is a self-contained offline exercise. Each device's change log must remain independent.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'dc-leaf-03'` |

**Logic to trace:** Use field(default_factory=list). Assert the second remains empty and the lists have different identities. Follow the class-specific steps under Pattern 032 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['reviewed'] []
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A factory is called for each needed default; passing one existing list explicitly can still share it.

In [13]:
# Example 0313: Fresh dataclass collection fields - dc
case = {'hostname': 'dc-leaf-03'}

from dataclasses import dataclass, field

@dataclass
class Device:
    hostname: str
    changes: list[str] = field(default_factory=list)

first = Device(case["hostname"])
second = Device("spare")
first.changes.append("reviewed")
assert second.changes == []
assert first.changes is not second.changes
print(first.changes, second.changes)


['reviewed'] []


### Example 0314 - Fresh dataclass collection fields: wan scenario

**Scenario 04/10; pattern 032; Intermediate.** This is a self-contained offline exercise. Each device's change log must remain independent.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'wan-border-04'` |

**Logic to trace:** Use field(default_factory=list). Assert the second remains empty and the lists have different identities. Follow the class-specific steps under Pattern 032 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['reviewed'] []
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A factory is called for each needed default; passing one existing list explicitly can still share it.

In [14]:
# Example 0314: Fresh dataclass collection fields - wan
case = {'hostname': 'wan-border-04'}

from dataclasses import dataclass, field

@dataclass
class Device:
    hostname: str
    changes: list[str] = field(default_factory=list)

first = Device(case["hostname"])
second = Device("spare")
first.changes.append("reviewed")
assert second.changes == []
assert first.changes is not second.changes
print(first.changes, second.changes)


['reviewed'] []


### Example 0315 - Fresh dataclass collection fields: lab scenario

**Scenario 05/10; pattern 032; Intermediate.** This is a self-contained offline exercise. Each device's change log must remain independent.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'lab-router-05'` |

**Logic to trace:** Use field(default_factory=list). Assert the second remains empty and the lists have different identities. Follow the class-specific steps under Pattern 032 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['reviewed'] []
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A factory is called for each needed default; passing one existing list explicitly can still share it.

In [15]:
# Example 0315: Fresh dataclass collection fields - lab
case = {'hostname': 'lab-router-05'}

from dataclasses import dataclass, field

@dataclass
class Device:
    hostname: str
    changes: list[str] = field(default_factory=list)

first = Device(case["hostname"])
second = Device("spare")
first.changes.append("reviewed")
assert second.changes == []
assert first.changes is not second.changes
print(first.changes, second.changes)


['reviewed'] []


### Example 0316 - Fresh dataclass collection fields: wireless scenario

**Scenario 06/10; pattern 032; Intermediate.** This is a self-contained offline exercise. Each device's change log must remain independent.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'wireless-dist-06'` |

**Logic to trace:** Use field(default_factory=list). Assert the second remains empty and the lists have different identities. Follow the class-specific steps under Pattern 032 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['reviewed'] []
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A factory is called for each needed default; passing one existing list explicitly can still share it.

In [16]:
# Example 0316: Fresh dataclass collection fields - wireless
case = {'hostname': 'wireless-dist-06'}

from dataclasses import dataclass, field

@dataclass
class Device:
    hostname: str
    changes: list[str] = field(default_factory=list)

first = Device(case["hostname"])
second = Device("spare")
first.changes.append("reviewed")
assert second.changes == []
assert first.changes is not second.changes
print(first.changes, second.changes)


['reviewed'] []


### Example 0317 - Fresh dataclass collection fields: remote scenario

**Scenario 07/10; pattern 032; Intermediate.** This is a self-contained offline exercise. Each device's change log must remain independent.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'remote-gateway-07'` |

**Logic to trace:** Use field(default_factory=list). Assert the second remains empty and the lists have different identities. Follow the class-specific steps under Pattern 032 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['reviewed'] []
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A factory is called for each needed default; passing one existing list explicitly can still share it.

In [17]:
# Example 0317: Fresh dataclass collection fields - remote
case = {'hostname': 'remote-gateway-07'}

from dataclasses import dataclass, field

@dataclass
class Device:
    hostname: str
    changes: list[str] = field(default_factory=list)

first = Device(case["hostname"])
second = Device("spare")
first.changes.append("reviewed")
assert second.changes == []
assert first.changes is not second.changes
print(first.changes, second.changes)


['reviewed'] []


### Example 0318 - Fresh dataclass collection fields: core scenario

**Scenario 08/10; pattern 032; Intermediate.** This is a self-contained offline exercise. Each device's change log must remain independent.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'core-router-08'` |

**Logic to trace:** Use field(default_factory=list). Assert the second remains empty and the lists have different identities. Follow the class-specific steps under Pattern 032 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['reviewed'] []
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A factory is called for each needed default; passing one existing list explicitly can still share it.

In [18]:
# Example 0318: Fresh dataclass collection fields - core
case = {'hostname': 'core-router-08'}

from dataclasses import dataclass, field

@dataclass
class Device:
    hostname: str
    changes: list[str] = field(default_factory=list)

first = Device(case["hostname"])
second = Device("spare")
first.changes.append("reviewed")
assert second.changes == []
assert first.changes is not second.changes
print(first.changes, second.changes)


['reviewed'] []


### Example 0319 - Fresh dataclass collection fields: mgmt scenario

**Scenario 09/10; pattern 032; Intermediate.** This is a self-contained offline exercise. Each device's change log must remain independent.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'mgmt-switch-09'` |

**Logic to trace:** Use field(default_factory=list). Assert the second remains empty and the lists have different identities. Follow the class-specific steps under Pattern 032 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['reviewed'] []
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A factory is called for each needed default; passing one existing list explicitly can still share it.

In [19]:
# Example 0319: Fresh dataclass collection fields - mgmt
case = {'hostname': 'mgmt-switch-09'}

from dataclasses import dataclass, field

@dataclass
class Device:
    hostname: str
    changes: list[str] = field(default_factory=list)

first = Device(case["hostname"])
second = Device("spare")
first.changes.append("reviewed")
assert second.changes == []
assert first.changes is not second.changes
print(first.changes, second.changes)


['reviewed'] []


### Example 0320 - Fresh dataclass collection fields: backup scenario

**Scenario 10/10; pattern 032; Intermediate.** This is a self-contained offline exercise. Each device's change log must remain independent.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'backup-edge-10'` |

**Logic to trace:** Use field(default_factory=list). Assert the second remains empty and the lists have different identities. Follow the class-specific steps under Pattern 032 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['reviewed'] []
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A factory is called for each needed default; passing one existing list explicitly can still share it.

In [20]:
# Example 0320: Fresh dataclass collection fields - backup
case = {'hostname': 'backup-edge-10'}

from dataclasses import dataclass, field

@dataclass
class Device:
    hostname: str
    changes: list[str] = field(default_factory=list)

first = Device(case["hostname"])
second = Device("spare")
first.changes.append("reviewed")
assert second.changes == []
assert first.changes is not second.changes
print(first.changes, second.changes)


['reviewed'] []


## Pattern 033: Frozen configuration and replace

**Core concept:** Immutable field assignment encourages explicit new desired-state values.

**Network-engineering purpose:** A proposed MTU can be represented without overwriting the approved baseline.

### Read the implementation in this order
1. Create a frozen dataclass.
2. Use replace to derive a changed record.
3. Check both old and new values.
4. Attempt direct mutation and verify the specific frozen-instance error.

**Common mistake / boundary:** Frozen is shallow; do not put mutable children in a value object you intend to hash.

**Practice after the ten worked scenarios:** Add a tuple of allowed VLANs and construct an updated value with replace().

The following ten examples intentionally reuse this implementation with explicit local scenario data. The displayed output comes from running each full code cell; assertions also check behavior, including expected rejections where present.

### Example 0321 - Frozen configuration and replace: branch scenario

**Scenario 01/10; pattern 033; Intermediate.** This is a self-contained offline exercise. A proposed MTU can be represented without overwriting the approved baseline.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `1500` |

**Logic to trace:** Create a frozen dataclass. Attempt direct mutation and verify the specific frozen-instance error. Follow the class-specific steps under Pattern 033 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
baseline protected
Config(mtu=1500)
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Frozen is shallow; do not put mutable children in a value object you intend to hash.

In [21]:
# Example 0321: Frozen configuration and replace - branch
case = {'mtu': 1500}

from dataclasses import FrozenInstanceError, dataclass, replace

@dataclass(frozen=True)
class Config:
    mtu: int

baseline = Config(1500)
proposed = replace(baseline, mtu=case["mtu"])
assert baseline.mtu == 1500 and proposed.mtu == case["mtu"]
try:
    baseline.mtu = 9000
except FrozenInstanceError:
    print("baseline protected")
else:
    raise AssertionError("frozen field changed")
print(proposed)


baseline protected
Config(mtu=1500)


### Example 0322 - Frozen configuration and replace: campus scenario

**Scenario 02/10; pattern 033; Intermediate.** This is a self-contained offline exercise. A proposed MTU can be represented without overwriting the approved baseline.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `9000` |

**Logic to trace:** Create a frozen dataclass. Attempt direct mutation and verify the specific frozen-instance error. Follow the class-specific steps under Pattern 033 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
baseline protected
Config(mtu=9000)
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Frozen is shallow; do not put mutable children in a value object you intend to hash.

In [22]:
# Example 0322: Frozen configuration and replace - campus
case = {'mtu': 9000}

from dataclasses import FrozenInstanceError, dataclass, replace

@dataclass(frozen=True)
class Config:
    mtu: int

baseline = Config(1500)
proposed = replace(baseline, mtu=case["mtu"])
assert baseline.mtu == 1500 and proposed.mtu == case["mtu"]
try:
    baseline.mtu = 9000
except FrozenInstanceError:
    print("baseline protected")
else:
    raise AssertionError("frozen field changed")
print(proposed)


baseline protected
Config(mtu=9000)


### Example 0323 - Frozen configuration and replace: dc scenario

**Scenario 03/10; pattern 033; Intermediate.** This is a self-contained offline exercise. A proposed MTU can be represented without overwriting the approved baseline.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `9216` |

**Logic to trace:** Create a frozen dataclass. Attempt direct mutation and verify the specific frozen-instance error. Follow the class-specific steps under Pattern 033 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
baseline protected
Config(mtu=9216)
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Frozen is shallow; do not put mutable children in a value object you intend to hash.

In [23]:
# Example 0323: Frozen configuration and replace - dc
case = {'mtu': 9216}

from dataclasses import FrozenInstanceError, dataclass, replace

@dataclass(frozen=True)
class Config:
    mtu: int

baseline = Config(1500)
proposed = replace(baseline, mtu=case["mtu"])
assert baseline.mtu == 1500 and proposed.mtu == case["mtu"]
try:
    baseline.mtu = 9000
except FrozenInstanceError:
    print("baseline protected")
else:
    raise AssertionError("frozen field changed")
print(proposed)


baseline protected
Config(mtu=9216)


### Example 0324 - Frozen configuration and replace: wan scenario

**Scenario 04/10; pattern 033; Intermediate.** This is a self-contained offline exercise. A proposed MTU can be represented without overwriting the approved baseline.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `576` |

**Logic to trace:** Create a frozen dataclass. Attempt direct mutation and verify the specific frozen-instance error. Follow the class-specific steps under Pattern 033 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
baseline protected
Config(mtu=576)
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Frozen is shallow; do not put mutable children in a value object you intend to hash.

In [24]:
# Example 0324: Frozen configuration and replace - wan
case = {'mtu': 576}

from dataclasses import FrozenInstanceError, dataclass, replace

@dataclass(frozen=True)
class Config:
    mtu: int

baseline = Config(1500)
proposed = replace(baseline, mtu=case["mtu"])
assert baseline.mtu == 1500 and proposed.mtu == case["mtu"]
try:
    baseline.mtu = 9000
except FrozenInstanceError:
    print("baseline protected")
else:
    raise AssertionError("frozen field changed")
print(proposed)


baseline protected
Config(mtu=576)


### Example 0325 - Frozen configuration and replace: lab scenario

**Scenario 05/10; pattern 033; Intermediate.** This is a self-contained offline exercise. A proposed MTU can be represented without overwriting the approved baseline.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `1600` |

**Logic to trace:** Create a frozen dataclass. Attempt direct mutation and verify the specific frozen-instance error. Follow the class-specific steps under Pattern 033 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
baseline protected
Config(mtu=1600)
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Frozen is shallow; do not put mutable children in a value object you intend to hash.

In [25]:
# Example 0325: Frozen configuration and replace - lab
case = {'mtu': 1600}

from dataclasses import FrozenInstanceError, dataclass, replace

@dataclass(frozen=True)
class Config:
    mtu: int

baseline = Config(1500)
proposed = replace(baseline, mtu=case["mtu"])
assert baseline.mtu == 1500 and proposed.mtu == case["mtu"]
try:
    baseline.mtu = 9000
except FrozenInstanceError:
    print("baseline protected")
else:
    raise AssertionError("frozen field changed")
print(proposed)


baseline protected
Config(mtu=1600)


### Example 0326 - Frozen configuration and replace: wireless scenario

**Scenario 06/10; pattern 033; Intermediate.** This is a self-contained offline exercise. A proposed MTU can be represented without overwriting the approved baseline.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `1500` |

**Logic to trace:** Create a frozen dataclass. Attempt direct mutation and verify the specific frozen-instance error. Follow the class-specific steps under Pattern 033 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
baseline protected
Config(mtu=1500)
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Frozen is shallow; do not put mutable children in a value object you intend to hash.

In [26]:
# Example 0326: Frozen configuration and replace - wireless
case = {'mtu': 1500}

from dataclasses import FrozenInstanceError, dataclass, replace

@dataclass(frozen=True)
class Config:
    mtu: int

baseline = Config(1500)
proposed = replace(baseline, mtu=case["mtu"])
assert baseline.mtu == 1500 and proposed.mtu == case["mtu"]
try:
    baseline.mtu = 9000
except FrozenInstanceError:
    print("baseline protected")
else:
    raise AssertionError("frozen field changed")
print(proposed)


baseline protected
Config(mtu=1500)


### Example 0327 - Frozen configuration and replace: remote scenario

**Scenario 07/10; pattern 033; Intermediate.** This is a self-contained offline exercise. A proposed MTU can be represented without overwriting the approved baseline.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `2000` |

**Logic to trace:** Create a frozen dataclass. Attempt direct mutation and verify the specific frozen-instance error. Follow the class-specific steps under Pattern 033 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
baseline protected
Config(mtu=2000)
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Frozen is shallow; do not put mutable children in a value object you intend to hash.

In [27]:
# Example 0327: Frozen configuration and replace - remote
case = {'mtu': 2000}

from dataclasses import FrozenInstanceError, dataclass, replace

@dataclass(frozen=True)
class Config:
    mtu: int

baseline = Config(1500)
proposed = replace(baseline, mtu=case["mtu"])
assert baseline.mtu == 1500 and proposed.mtu == case["mtu"]
try:
    baseline.mtu = 9000
except FrozenInstanceError:
    print("baseline protected")
else:
    raise AssertionError("frozen field changed")
print(proposed)


baseline protected


Config(mtu=2000)


### Example 0328 - Frozen configuration and replace: core scenario

**Scenario 08/10; pattern 033; Intermediate.** This is a self-contained offline exercise. A proposed MTU can be represented without overwriting the approved baseline.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `4470` |

**Logic to trace:** Create a frozen dataclass. Attempt direct mutation and verify the specific frozen-instance error. Follow the class-specific steps under Pattern 033 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
baseline protected
Config(mtu=4470)
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Frozen is shallow; do not put mutable children in a value object you intend to hash.

In [28]:
# Example 0328: Frozen configuration and replace - core
case = {'mtu': 4470}

from dataclasses import FrozenInstanceError, dataclass, replace

@dataclass(frozen=True)
class Config:
    mtu: int

baseline = Config(1500)
proposed = replace(baseline, mtu=case["mtu"])
assert baseline.mtu == 1500 and proposed.mtu == case["mtu"]
try:
    baseline.mtu = 9000
except FrozenInstanceError:
    print("baseline protected")
else:
    raise AssertionError("frozen field changed")
print(proposed)


baseline protected
Config(mtu=4470)


### Example 0329 - Frozen configuration and replace: mgmt scenario

**Scenario 09/10; pattern 033; Intermediate.** This is a self-contained offline exercise. A proposed MTU can be represented without overwriting the approved baseline.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `9100` |

**Logic to trace:** Create a frozen dataclass. Attempt direct mutation and verify the specific frozen-instance error. Follow the class-specific steps under Pattern 033 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
baseline protected
Config(mtu=9100)
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Frozen is shallow; do not put mutable children in a value object you intend to hash.

In [29]:
# Example 0329: Frozen configuration and replace - mgmt
case = {'mtu': 9100}

from dataclasses import FrozenInstanceError, dataclass, replace

@dataclass(frozen=True)
class Config:
    mtu: int

baseline = Config(1500)
proposed = replace(baseline, mtu=case["mtu"])
assert baseline.mtu == 1500 and proposed.mtu == case["mtu"]
try:
    baseline.mtu = 9000
except FrozenInstanceError:
    print("baseline protected")
else:
    raise AssertionError("frozen field changed")
print(proposed)


baseline protected
Config(mtu=9100)


### Example 0330 - Frozen configuration and replace: backup scenario

**Scenario 10/10; pattern 033; Intermediate.** This is a self-contained offline exercise. A proposed MTU can be represented without overwriting the approved baseline.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `8192` |

**Logic to trace:** Create a frozen dataclass. Attempt direct mutation and verify the specific frozen-instance error. Follow the class-specific steps under Pattern 033 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
baseline protected
Config(mtu=8192)
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Frozen is shallow; do not put mutable children in a value object you intend to hash.

In [30]:
# Example 0330: Frozen configuration and replace - backup
case = {'mtu': 8192}

from dataclasses import FrozenInstanceError, dataclass, replace

@dataclass(frozen=True)
class Config:
    mtu: int

baseline = Config(1500)
proposed = replace(baseline, mtu=case["mtu"])
assert baseline.mtu == 1500 and proposed.mtu == case["mtu"]
try:
    baseline.mtu = 9000
except FrozenInstanceError:
    print("baseline protected")
else:
    raise AssertionError("frozen field changed")
print(proposed)


baseline protected
Config(mtu=8192)


## Pattern 034: Explicit dataclass ordering

**Core concept:** Ordered records compare declared comparison fields in sequence.

**Network-engineering purpose:** A small preference model can choose the lowest local ranking before a tie-break label.

### Read the implementation in this order
1. Place preference before name in the field order.
2. Enable order=True.
3. Sort two candidates.
4. Verify the lower preference wins.

**Common mistake / boundary:** This local ranking is not a complete routing protocol best-path algorithm.

**Practice after the ten worked scenarios:** Use field(compare=False) for a description that should not affect selection.

The following ten examples intentionally reuse this implementation with explicit local scenario data. The displayed output comes from running each full code cell; assertions also check behavior, including expected rejections where present.

### Example 0331 - Explicit dataclass ordering: branch scenario

**Scenario 01/10; pattern 034; Intermediate.** This is a self-contained offline exercise. A small preference model can choose the lowest local ranking before a tie-break label.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `cost` | `10` |

**Logic to trace:** Place preference before name in the field order. Verify the lower preference wins. Follow the class-specific steps under Pattern 034 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
Candidate(preference=10, name='primary')
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This local ranking is not a complete routing protocol best-path algorithm.

In [31]:
# Example 0331: Explicit dataclass ordering - branch
case = {'cost': 10}

from dataclasses import dataclass

@dataclass(order=True)
class Candidate:
    preference: int
    name: str

candidates = [Candidate(case["cost"] + 10, "backup"), Candidate(case["cost"], "primary")]
selected = sorted(candidates)[0]
assert selected.name == "primary"
print(selected)


Candidate(preference=10, name='primary')


### Example 0332 - Explicit dataclass ordering: campus scenario

**Scenario 02/10; pattern 034; Intermediate.** This is a self-contained offline exercise. A small preference model can choose the lowest local ranking before a tie-break label.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `cost` | `20` |

**Logic to trace:** Place preference before name in the field order. Verify the lower preference wins. Follow the class-specific steps under Pattern 034 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
Candidate(preference=20, name='primary')
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This local ranking is not a complete routing protocol best-path algorithm.

In [32]:
# Example 0332: Explicit dataclass ordering - campus
case = {'cost': 20}

from dataclasses import dataclass

@dataclass(order=True)
class Candidate:
    preference: int
    name: str

candidates = [Candidate(case["cost"] + 10, "backup"), Candidate(case["cost"], "primary")]
selected = sorted(candidates)[0]
assert selected.name == "primary"
print(selected)


Candidate(preference=20, name='primary')


### Example 0333 - Explicit dataclass ordering: dc scenario

**Scenario 03/10; pattern 034; Intermediate.** This is a self-contained offline exercise. A small preference model can choose the lowest local ranking before a tie-break label.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `cost` | `30` |

**Logic to trace:** Place preference before name in the field order. Verify the lower preference wins. Follow the class-specific steps under Pattern 034 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
Candidate(preference=30, name='primary')
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This local ranking is not a complete routing protocol best-path algorithm.

In [33]:
# Example 0333: Explicit dataclass ordering - dc
case = {'cost': 30}

from dataclasses import dataclass

@dataclass(order=True)
class Candidate:
    preference: int
    name: str

candidates = [Candidate(case["cost"] + 10, "backup"), Candidate(case["cost"], "primary")]
selected = sorted(candidates)[0]
assert selected.name == "primary"
print(selected)


Candidate(preference=30, name='primary')


### Example 0334 - Explicit dataclass ordering: wan scenario

**Scenario 04/10; pattern 034; Intermediate.** This is a self-contained offline exercise. A small preference model can choose the lowest local ranking before a tie-break label.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `cost` | `40` |

**Logic to trace:** Place preference before name in the field order. Verify the lower preference wins. Follow the class-specific steps under Pattern 034 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
Candidate(preference=40, name='primary')
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This local ranking is not a complete routing protocol best-path algorithm.

In [34]:
# Example 0334: Explicit dataclass ordering - wan
case = {'cost': 40}

from dataclasses import dataclass

@dataclass(order=True)
class Candidate:
    preference: int
    name: str

candidates = [Candidate(case["cost"] + 10, "backup"), Candidate(case["cost"], "primary")]
selected = sorted(candidates)[0]
assert selected.name == "primary"
print(selected)


Candidate(preference=40, name='primary')


### Example 0335 - Explicit dataclass ordering: lab scenario

**Scenario 05/10; pattern 034; Intermediate.** This is a self-contained offline exercise. A small preference model can choose the lowest local ranking before a tie-break label.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `cost` | `50` |

**Logic to trace:** Place preference before name in the field order. Verify the lower preference wins. Follow the class-specific steps under Pattern 034 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
Candidate(preference=50, name='primary')
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This local ranking is not a complete routing protocol best-path algorithm.

In [35]:
# Example 0335: Explicit dataclass ordering - lab
case = {'cost': 50}

from dataclasses import dataclass

@dataclass(order=True)
class Candidate:
    preference: int
    name: str

candidates = [Candidate(case["cost"] + 10, "backup"), Candidate(case["cost"], "primary")]
selected = sorted(candidates)[0]
assert selected.name == "primary"
print(selected)


Candidate(preference=50, name='primary')


### Example 0336 - Explicit dataclass ordering: wireless scenario

**Scenario 06/10; pattern 034; Intermediate.** This is a self-contained offline exercise. A small preference model can choose the lowest local ranking before a tie-break label.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `cost` | `60` |

**Logic to trace:** Place preference before name in the field order. Verify the lower preference wins. Follow the class-specific steps under Pattern 034 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
Candidate(preference=60, name='primary')
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This local ranking is not a complete routing protocol best-path algorithm.

In [36]:
# Example 0336: Explicit dataclass ordering - wireless
case = {'cost': 60}

from dataclasses import dataclass

@dataclass(order=True)
class Candidate:
    preference: int
    name: str

candidates = [Candidate(case["cost"] + 10, "backup"), Candidate(case["cost"], "primary")]
selected = sorted(candidates)[0]
assert selected.name == "primary"
print(selected)


Candidate(preference=60, name='primary')


### Example 0337 - Explicit dataclass ordering: remote scenario

**Scenario 07/10; pattern 034; Intermediate.** This is a self-contained offline exercise. A small preference model can choose the lowest local ranking before a tie-break label.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `cost` | `70` |

**Logic to trace:** Place preference before name in the field order. Verify the lower preference wins. Follow the class-specific steps under Pattern 034 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
Candidate(preference=70, name='primary')
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This local ranking is not a complete routing protocol best-path algorithm.

In [37]:
# Example 0337: Explicit dataclass ordering - remote
case = {'cost': 70}

from dataclasses import dataclass

@dataclass(order=True)
class Candidate:
    preference: int
    name: str

candidates = [Candidate(case["cost"] + 10, "backup"), Candidate(case["cost"], "primary")]
selected = sorted(candidates)[0]
assert selected.name == "primary"
print(selected)


Candidate(preference=70, name='primary')


### Example 0338 - Explicit dataclass ordering: core scenario

**Scenario 08/10; pattern 034; Intermediate.** This is a self-contained offline exercise. A small preference model can choose the lowest local ranking before a tie-break label.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `cost` | `80` |

**Logic to trace:** Place preference before name in the field order. Verify the lower preference wins. Follow the class-specific steps under Pattern 034 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
Candidate(preference=80, name='primary')
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This local ranking is not a complete routing protocol best-path algorithm.

In [38]:
# Example 0338: Explicit dataclass ordering - core
case = {'cost': 80}

from dataclasses import dataclass

@dataclass(order=True)
class Candidate:
    preference: int
    name: str

candidates = [Candidate(case["cost"] + 10, "backup"), Candidate(case["cost"], "primary")]
selected = sorted(candidates)[0]
assert selected.name == "primary"
print(selected)


Candidate(preference=80, name='primary')


### Example 0339 - Explicit dataclass ordering: mgmt scenario

**Scenario 09/10; pattern 034; Intermediate.** This is a self-contained offline exercise. A small preference model can choose the lowest local ranking before a tie-break label.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `cost` | `90` |

**Logic to trace:** Place preference before name in the field order. Verify the lower preference wins. Follow the class-specific steps under Pattern 034 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
Candidate(preference=90, name='primary')
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This local ranking is not a complete routing protocol best-path algorithm.

In [39]:
# Example 0339: Explicit dataclass ordering - mgmt
case = {'cost': 90}

from dataclasses import dataclass

@dataclass(order=True)
class Candidate:
    preference: int
    name: str

candidates = [Candidate(case["cost"] + 10, "backup"), Candidate(case["cost"], "primary")]
selected = sorted(candidates)[0]
assert selected.name == "primary"
print(selected)


Candidate(preference=90, name='primary')


### Example 0340 - Explicit dataclass ordering: backup scenario

**Scenario 10/10; pattern 034; Intermediate.** This is a self-contained offline exercise. A small preference model can choose the lowest local ranking before a tie-break label.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `cost` | `100` |

**Logic to trace:** Place preference before name in the field order. Verify the lower preference wins. Follow the class-specific steps under Pattern 034 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
Candidate(preference=100, name='primary')
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This local ranking is not a complete routing protocol best-path algorithm.

In [40]:
# Example 0340: Explicit dataclass ordering - backup
case = {'cost': 100}

from dataclasses import dataclass

@dataclass(order=True)
class Candidate:
    preference: int
    name: str

candidates = [Candidate(case["cost"] + 10, "backup"), Candidate(case["cost"], "primary")]
selected = sorted(candidates)[0]
assert selected.name == "primary"
print(selected)


Candidate(preference=100, name='primary')


## Pattern 035: Shallow versus deep copy

**Core concept:** Copy depth determines which nested objects remain shared.

**Network-engineering purpose:** A candidate inventory snapshot should not accidentally mutate a baseline interface list.

### Read the implementation in this order
1. Build a dataclass with a mutable child list.
2. Make a shallow and a deep copy.
3. Mutate the shallow copy's child.
4. Check the baseline changed but the deep snapshot did not.

**Common mistake / boundary:** Deep-copy configuration data, not live connection handles.

**Practice after the ten worked scenarios:** Replace the list with a tuple and compare the need for defensive copying.

The following ten examples intentionally reuse this implementation with explicit local scenario data. The displayed output comes from running each full code cell; assertions also check behavior, including expected rejections where present.

### Example 0341 - Shallow versus deep copy: branch scenario

**Scenario 01/10; pattern 035; Intermediate.** This is a self-contained offline exercise. A candidate inventory snapshot should not accidentally mutate a baseline interface list.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `interface` | `'Ethernet1/1'` |

**Logic to trace:** Build a dataclass with a mutable child list. Check the baseline changed but the deep snapshot did not. Follow the class-specific steps under Pattern 035 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
baseline: ['Ethernet1/1', 'Loopback7'] snapshot: ['Ethernet1/1']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Deep-copy configuration data, not live connection handles.

In [41]:
# Example 0341: Shallow versus deep copy - branch
case = {'interface': 'Ethernet1/1'}

from copy import copy, deepcopy
from dataclasses import dataclass

@dataclass
class Inventory:
    interfaces: list[str]

original = Inventory([case["interface"]])
shallow, deep = copy(original), deepcopy(original)
shallow.interfaces.append("Loopback7")
assert original.interfaces == [case["interface"], "Loopback7"]
assert deep.interfaces == [case["interface"]]
assert shallow is not original
print("baseline:", original.interfaces, "snapshot:", deep.interfaces)


baseline: ['Ethernet1/1', 'Loopback7'] snapshot: ['Ethernet1/1']


### Example 0342 - Shallow versus deep copy: campus scenario

**Scenario 02/10; pattern 035; Intermediate.** This is a self-contained offline exercise. A candidate inventory snapshot should not accidentally mutate a baseline interface list.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `interface` | `'Ethernet2/1'` |

**Logic to trace:** Build a dataclass with a mutable child list. Check the baseline changed but the deep snapshot did not. Follow the class-specific steps under Pattern 035 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
baseline: ['Ethernet2/1', 'Loopback7'] snapshot: ['Ethernet2/1']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Deep-copy configuration data, not live connection handles.

In [42]:
# Example 0342: Shallow versus deep copy - campus
case = {'interface': 'Ethernet2/1'}

from copy import copy, deepcopy
from dataclasses import dataclass

@dataclass
class Inventory:
    interfaces: list[str]

original = Inventory([case["interface"]])
shallow, deep = copy(original), deepcopy(original)
shallow.interfaces.append("Loopback7")
assert original.interfaces == [case["interface"], "Loopback7"]
assert deep.interfaces == [case["interface"]]
assert shallow is not original
print("baseline:", original.interfaces, "snapshot:", deep.interfaces)


baseline: ['Ethernet2/1', 'Loopback7'] snapshot: ['Ethernet2/1']


### Example 0343 - Shallow versus deep copy: dc scenario

**Scenario 03/10; pattern 035; Intermediate.** This is a self-contained offline exercise. A candidate inventory snapshot should not accidentally mutate a baseline interface list.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `interface` | `'Ethernet3/1'` |

**Logic to trace:** Build a dataclass with a mutable child list. Check the baseline changed but the deep snapshot did not. Follow the class-specific steps under Pattern 035 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
baseline: ['Ethernet3/1', 'Loopback7'] snapshot: ['Ethernet3/1']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Deep-copy configuration data, not live connection handles.

In [43]:
# Example 0343: Shallow versus deep copy - dc
case = {'interface': 'Ethernet3/1'}

from copy import copy, deepcopy
from dataclasses import dataclass

@dataclass
class Inventory:
    interfaces: list[str]

original = Inventory([case["interface"]])
shallow, deep = copy(original), deepcopy(original)
shallow.interfaces.append("Loopback7")
assert original.interfaces == [case["interface"], "Loopback7"]
assert deep.interfaces == [case["interface"]]
assert shallow is not original
print("baseline:", original.interfaces, "snapshot:", deep.interfaces)


baseline: ['Ethernet3/1', 'Loopback7'] snapshot: ['Ethernet3/1']


### Example 0344 - Shallow versus deep copy: wan scenario

**Scenario 04/10; pattern 035; Intermediate.** This is a self-contained offline exercise. A candidate inventory snapshot should not accidentally mutate a baseline interface list.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `interface` | `'Ethernet4/1'` |

**Logic to trace:** Build a dataclass with a mutable child list. Check the baseline changed but the deep snapshot did not. Follow the class-specific steps under Pattern 035 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
baseline: ['Ethernet4/1', 'Loopback7'] snapshot: ['Ethernet4/1']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Deep-copy configuration data, not live connection handles.

In [44]:
# Example 0344: Shallow versus deep copy - wan
case = {'interface': 'Ethernet4/1'}

from copy import copy, deepcopy
from dataclasses import dataclass

@dataclass
class Inventory:
    interfaces: list[str]

original = Inventory([case["interface"]])
shallow, deep = copy(original), deepcopy(original)
shallow.interfaces.append("Loopback7")
assert original.interfaces == [case["interface"], "Loopback7"]
assert deep.interfaces == [case["interface"]]
assert shallow is not original
print("baseline:", original.interfaces, "snapshot:", deep.interfaces)


baseline: ['Ethernet4/1', 'Loopback7'] snapshot: ['Ethernet4/1']


### Example 0345 - Shallow versus deep copy: lab scenario

**Scenario 05/10; pattern 035; Intermediate.** This is a self-contained offline exercise. A candidate inventory snapshot should not accidentally mutate a baseline interface list.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `interface` | `'Ethernet5/1'` |

**Logic to trace:** Build a dataclass with a mutable child list. Check the baseline changed but the deep snapshot did not. Follow the class-specific steps under Pattern 035 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
baseline: ['Ethernet5/1', 'Loopback7'] snapshot: ['Ethernet5/1']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Deep-copy configuration data, not live connection handles.

In [45]:
# Example 0345: Shallow versus deep copy - lab
case = {'interface': 'Ethernet5/1'}

from copy import copy, deepcopy
from dataclasses import dataclass

@dataclass
class Inventory:
    interfaces: list[str]

original = Inventory([case["interface"]])
shallow, deep = copy(original), deepcopy(original)
shallow.interfaces.append("Loopback7")
assert original.interfaces == [case["interface"], "Loopback7"]
assert deep.interfaces == [case["interface"]]
assert shallow is not original
print("baseline:", original.interfaces, "snapshot:", deep.interfaces)


baseline: ['Ethernet5/1', 'Loopback7'] snapshot: ['Ethernet5/1']


### Example 0346 - Shallow versus deep copy: wireless scenario

**Scenario 06/10; pattern 035; Intermediate.** This is a self-contained offline exercise. A candidate inventory snapshot should not accidentally mutate a baseline interface list.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `interface` | `'Ethernet6/1'` |

**Logic to trace:** Build a dataclass with a mutable child list. Check the baseline changed but the deep snapshot did not. Follow the class-specific steps under Pattern 035 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
baseline: ['Ethernet6/1', 'Loopback7'] snapshot: ['Ethernet6/1']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Deep-copy configuration data, not live connection handles.

In [46]:
# Example 0346: Shallow versus deep copy - wireless
case = {'interface': 'Ethernet6/1'}

from copy import copy, deepcopy
from dataclasses import dataclass

@dataclass
class Inventory:
    interfaces: list[str]

original = Inventory([case["interface"]])
shallow, deep = copy(original), deepcopy(original)
shallow.interfaces.append("Loopback7")
assert original.interfaces == [case["interface"], "Loopback7"]
assert deep.interfaces == [case["interface"]]
assert shallow is not original
print("baseline:", original.interfaces, "snapshot:", deep.interfaces)


baseline: ['Ethernet6/1', 'Loopback7'] snapshot: ['Ethernet6/1']


### Example 0347 - Shallow versus deep copy: remote scenario

**Scenario 07/10; pattern 035; Intermediate.** This is a self-contained offline exercise. A candidate inventory snapshot should not accidentally mutate a baseline interface list.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `interface` | `'Ethernet7/1'` |

**Logic to trace:** Build a dataclass with a mutable child list. Check the baseline changed but the deep snapshot did not. Follow the class-specific steps under Pattern 035 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
baseline: ['Ethernet7/1', 'Loopback7'] snapshot: ['Ethernet7/1']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Deep-copy configuration data, not live connection handles.

In [47]:
# Example 0347: Shallow versus deep copy - remote
case = {'interface': 'Ethernet7/1'}

from copy import copy, deepcopy
from dataclasses import dataclass

@dataclass
class Inventory:
    interfaces: list[str]

original = Inventory([case["interface"]])
shallow, deep = copy(original), deepcopy(original)
shallow.interfaces.append("Loopback7")
assert original.interfaces == [case["interface"], "Loopback7"]
assert deep.interfaces == [case["interface"]]
assert shallow is not original
print("baseline:", original.interfaces, "snapshot:", deep.interfaces)


baseline: ['Ethernet7/1', 'Loopback7'] snapshot: ['Ethernet7/1']


### Example 0348 - Shallow versus deep copy: core scenario

**Scenario 08/10; pattern 035; Intermediate.** This is a self-contained offline exercise. A candidate inventory snapshot should not accidentally mutate a baseline interface list.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `interface` | `'Ethernet8/1'` |

**Logic to trace:** Build a dataclass with a mutable child list. Check the baseline changed but the deep snapshot did not. Follow the class-specific steps under Pattern 035 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
baseline: ['Ethernet8/1', 'Loopback7'] snapshot: ['Ethernet8/1']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Deep-copy configuration data, not live connection handles.

In [48]:
# Example 0348: Shallow versus deep copy - core
case = {'interface': 'Ethernet8/1'}

from copy import copy, deepcopy
from dataclasses import dataclass

@dataclass
class Inventory:
    interfaces: list[str]

original = Inventory([case["interface"]])
shallow, deep = copy(original), deepcopy(original)
shallow.interfaces.append("Loopback7")
assert original.interfaces == [case["interface"], "Loopback7"]
assert deep.interfaces == [case["interface"]]
assert shallow is not original
print("baseline:", original.interfaces, "snapshot:", deep.interfaces)


baseline: ['Ethernet8/1', 'Loopback7'] snapshot: ['Ethernet8/1']


### Example 0349 - Shallow versus deep copy: mgmt scenario

**Scenario 09/10; pattern 035; Intermediate.** This is a self-contained offline exercise. A candidate inventory snapshot should not accidentally mutate a baseline interface list.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `interface` | `'Ethernet9/1'` |

**Logic to trace:** Build a dataclass with a mutable child list. Check the baseline changed but the deep snapshot did not. Follow the class-specific steps under Pattern 035 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
baseline: ['Ethernet9/1', 'Loopback7'] snapshot: ['Ethernet9/1']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Deep-copy configuration data, not live connection handles.

In [49]:
# Example 0349: Shallow versus deep copy - mgmt
case = {'interface': 'Ethernet9/1'}

from copy import copy, deepcopy
from dataclasses import dataclass

@dataclass
class Inventory:
    interfaces: list[str]

original = Inventory([case["interface"]])
shallow, deep = copy(original), deepcopy(original)
shallow.interfaces.append("Loopback7")
assert original.interfaces == [case["interface"], "Loopback7"]
assert deep.interfaces == [case["interface"]]
assert shallow is not original
print("baseline:", original.interfaces, "snapshot:", deep.interfaces)


baseline: ['Ethernet9/1', 'Loopback7'] snapshot: ['Ethernet9/1']


### Example 0350 - Shallow versus deep copy: backup scenario

**Scenario 10/10; pattern 035; Intermediate.** This is a self-contained offline exercise. A candidate inventory snapshot should not accidentally mutate a baseline interface list.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `interface` | `'Ethernet10/1'` |

**Logic to trace:** Build a dataclass with a mutable child list. Check the baseline changed but the deep snapshot did not. Follow the class-specific steps under Pattern 035 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
baseline: ['Ethernet10/1', 'Loopback7'] snapshot: ['Ethernet10/1']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Deep-copy configuration data, not live connection handles.

In [50]:
# Example 0350: Shallow versus deep copy - backup
case = {'interface': 'Ethernet10/1'}

from copy import copy, deepcopy
from dataclasses import dataclass

@dataclass
class Inventory:
    interfaces: list[str]

original = Inventory([case["interface"]])
shallow, deep = copy(original), deepcopy(original)
shallow.interfaces.append("Loopback7")
assert original.interfaces == [case["interface"], "Loopback7"]
assert deep.interfaces == [case["interface"]]
assert shallow is not original
print("baseline:", original.interfaces, "snapshot:", deep.interfaces)


baseline: ['Ethernet10/1', 'Loopback7'] snapshot: ['Ethernet10/1']


## Check your understanding: Dataclasses, value objects, and copying

**You should now be able to:** Choose field-based equality, immutable records, factories, ordering, and appropriate copy depth.

1. Explain which object owns each piece of state and which operations can change it.
2. Walk through a normal result and an invalid-input or failure path without running the code.
3. Name one invariant, one useful assertion, and one production assumption not proved by the fake.
4. Implement one extension prompt, including a test that would fail if your new behavior were wrong.
5. Explain when a function or a dataclass would be simpler than another subclass or framework.

**Do not confuse:** a successful local assertion with real device compatibility; missing data with a healthy result; underscore naming with access control; type annotations with runtime validation; or local rollback with a multi-device transaction.

## Official reference reading
- `https://docs.python.org/3.13/tutorial/classes.html`
- `https://docs.python.org/3.13/reference/datamodel.html`

References describe Python contracts. They do not certify any vendor command or network policy.